# Inventory dynamics — code from the class

Every code example from the *Dynamic programming* chapter, <https://dse.iskh.me/dp>, in
the order it appears there: the deterministic inventory model in finite horizon, solved
by backwards induction, and the demand distribution of the stochastic version.

Run the cells top to bottom — later ones use names defined earlier.

### Bellman equation for the problem

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = [12, 8]

class inventory_model:
    '''Small class to hold model fundamentals and its solution'''

    def __init__(self,label='noname',
                 max_inventory=10,  # upper bound on the state space
                 c = 3.2,           # fixed cost of order
                 p = 2.5,           # profit per unit of good
                 r = 0.5,           # storage cost per unit of good
                 β = 0.95,          # discount factor
                 demand = 4         # fixed demand
                 ):
        '''Create model with default parameters'''
        self.label=label # label for the model instance
        self.c, self.p, self.r, self.β = c, p, r, β
        self.demand = demand
        # created dependent attributes (it would be better to have them updated when underlying parameters change)
        self.n = max_inventory+1    # number of inventory levels
        self.upper = max_inventory  # upper boundary on inventory
        self.x = np.arange(self.n)  # all possible values of inventory (state space)

    def __repr__(self):
        '''String representation of the model'''
        return 'Inventory model labeled "{}"\nParamters (c,p,r,β) = ({},{},{},{})\nDemand={}\nUpper bound on inventory {}' \
               .format (self.label,self.c,self.p,self.r,self.β,self.demand,self.upper)

    def sales(self,x,d):
        '''Sales in given period'''
        return np.minimum(x,d)

    def next_x(self,x,d,q):
        '''Inventory to be stored, becomes next period state'''
        return x - self.sales(x,d) + q

    def profit(self,x,d,q):
        '''Profit in given period'''
        return self.p * self.sales(x,d) - self.r * self.next_x(x,d,q) - self.c * (q>0)

In [ ]:
model=inventory_model(label='test')
print(model)

q=np.zeros(model.n)
print('Current profits with zero orders\n',model.profit(model.x,model.demand,q))

In [ ]:
# illustration of broadcasting in the inventory model
q=model.x[:,np.newaxis]  # column vector
print('Current inventory\n',model.x)
print('Current sales\n',model.sales(model.x,model.demand))
print('Current orders\n',q)
print('Next period inventory\n',model.next_x(model.x,model.demand,q))
print('Current profits\n',model.profit(model.x,model.demand,q))

## Backwards induction

In [ ]:
def bellman(m,v0):
    '''Bellman equation for inventory model
       Inputs: model object
               next period value function
    '''
    # create the grid of choices (same as x), column-vector
    q = m.x[:,np.newaxis]
    # compute current period profit (relying on numpy broadcasting to get the matrix with choices in rows)
    p = m.profit(m.x,m.demand,q)
    # indexes for next period value with extrapolation using last value
    i = np.minimum(m.next_x(m.x,m.demand,q),m.upper)
    # compute the Bellman maximand
    vm = p + m.β*v0[i]
    # find max and argmax
    v1 = np.amax(vm,axis=0)   # maximum in every column
    q1 = np.argmax(vm,axis=0) # arg-maximum in every column = order volume
    return v1, q1

In [ ]:
v = np.zeros(model.n)
for i in range(3):
    v,q = bellman(model,v)
    print('Value =',v,'Policy =',q,sep='\n',end='\n\n')

In [ ]:
def solver_backwards_induction(m,T=10,verbose=False):
    '''Backwards induction solver for the finite horizon case'''
    # solution is time dependent
    m.value  = np.zeros((m.n,T))
    m.policy = np.zeros((m.n,T))
    # main DP loop (from T to 1)
    for t in range(T,0,-1):
        if verbose:
            print('Time period %d\n'%t)
        j = t-1 # index of value and policy functions for period t
        if t==T:
            # terminal period: ordering zero is optimal
            m.value[:,j] = m.profit(m.x,m.demand,np.zeros(m.n))
            m.policy[:,j] = np.zeros(m.n)
        else:
            # all other periods
            m.value[:,j], m.policy[:,j] = bellman(m,m.value[:,j+1]) # next period to Bellman
        if verbose:
            print(m.value,'\n')
    # return model with updated value and policy functions
    return m

model = inventory_model(label='illustration')
model=solver_backwards_induction(model,T=5,verbose=True)
print('Optimal policy:\n',model.policy)

In [ ]:
def plot_solution(model):
    plt.step(model.x,model.value)
    plt.legend([f'{i+1}' for i in range(model.value.shape[1])])
    plt.title('Value function')
    plt.show()
    plt.step(model.x,model.policy)
    plt.legend([f'{i+1}' for i in range(model.policy.shape[1])])
    plt.title('Policy function (optimal order sizes)')
    plt.show()
plot_solution(model)

In [ ]:
mod = inventory_model(label='production',max_inventory=50)
mod.demand = 15
mod.c = 5
mod.p = 2.5
mod.r = 1.4
mod.β = 0.975
mod = solver_backwards_induction(mod,T=15)
plot_solution(mod)

## Back to stochastic demand

In [ ]:
N = 25                                    # upper bound of the inventory grid
k = np.arange(N+1)                        # possible values of demand

def demand_pr(lam, n=N+1):
    '''Truncated geometric probabilities over the grid, last one corrected'''
    pr = (1-lam)**np.arange(n) * lam
    pr[-1] = 1 - pr[:-1].sum()            # the truncated tail piles onto the last point
    return pr

rng = np.random.default_rng(2026)
fig, (ax1,ax2) = plt.subplots(1,2,figsize=(12,4))
for lam,clr in [(0.15,'#1A6D91'), (0.25,'#0E8A27'), (0.50,'#CB1515')]:
    pr = demand_pr(lam)
    ax1.step(k, pr, where='mid', color=clr, lw=2,
             label=f'$\\lambda$={lam:.2f}, mean {np.dot(k,pr):.1f}, $pr_N$={pr[-1]:.3f}')
ax1.set_title('Truncated geometric demand')
ax1.set_xlabel('demand $d$'); ax1.set_ylabel('probability')
ax1.legend(frameon=False); ax1.grid(alpha=.3)

lam, T = 0.25, 40
pr = demand_pr(lam)
d = rng.choice(k, size=T, p=pr)           # one path of realized demand
ax2.step(np.arange(T), d, where='mid', color='#0E8A27', lw=1.5)
ax2.axhline(np.dot(k,pr), color='k', ls='--', lw=1, label='mean demand')
ax2.set_title(f'One realization over {T} periods, $\\lambda$={lam:.2f}')
ax2.set_xlabel('period $t$'); ax2.set_ylabel('demand $d_t$')
ax2.legend(frameon=False); ax2.grid(alpha=.3)
plt.show()